# Three-Stream Propaganda Detection
**Neural + Knowledge Graph + LLM Architecture**

This notebook demonstrates the complete three-stream architecture for detecting propaganda techniques in memes.

## Architecture Overview

```
                    Meme (Text + Image)
                            |
        ┌───────────────────┼───────────────────┐
        │                   │                   │
   Stream 1             Stream 2           Stream 3
   Neural            Knowledge Graph        LLM
Pattern Recognition  Rule-Based Reasoning  Verification
(CLIP + RoBERTa)     (Prerequisites +      (GPT-4/Claude)
                      Co-occurrence)
        │                   │                   │
        └───────────────────┼───────────────────┘
                            │
                        Fusion
                            │
                   Final Predictions
```

## Notebook Sections

1. **Setup**: Mount Drive, install packages, import modules
2. **Configuration**: Set paths and hyperparameters
3. **Option A: Inference Only** (use pretrained model)
4. **Option B: Training** (train neural stream from scratch)
5. **Evaluation**: Test on validation/test set
6. **Analysis**: Visualize results and errors

---
## 1. Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install required packages
!pip install -q transformers torch torchvision pillow networkx scikit-learn tqdm

In [ ]:
# Navigate to project directory
import os
import sys

# UPDATE THIS PATH to your project folder in Drive
PROJECT_DIR = "/content/drive/MyDrive/propaganda_detection"

os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)

print(f"Working directory: {os.getcwd()}")
print(f"Files: {os.listdir('.')}")

In [ ]:
# Import our modules
from config import ExperimentConfig, get_config_colab_no_llm
from pipeline import ThreeStreamPropagandaDetector
from trainer import NeuralStreamTrainer
from data_loader import create_label_mappings, create_dataloaders, PropagandaDataLoader

import torch
import pandas as pd
import json
from pathlib import Path

print("✓ All modules imported successfully")

---
## 2. Configuration

Set your paths and hyperparameters here.

In [ ]:
# Create configuration
config = get_config_colab_no_llm()  # Neural + KG only (no LLM cost)

# Update paths if needed
config.data.drive_root = "/content/drive/MyDrive"
config.data.project_dir = "propaganda_detection"

# Data files (relative to project_dir or absolute paths)
config.data.train_json = "dataset_with_rationales_subtask2a_final (1).json"
config.data.val_json = "validation_caption.json"
config.data.test_json = "dev_processed.json"

# Image directories (absolute paths)
config.data.train_img_dir = "/content/train_images/train_images"
config.data.val_img_dir = "/content/validation_images/validation_images"
config.data.test_img_dir = "/content/dev_images/dev_images"

# Training settings
config.neural.batch_size = 32
config.neural.epochs = 12
config.neural.learning_rate = 2e-3

# Fusion weights (can optimize later)
config.fusion.alpha = 0.5  # Neural weight
config.fusion.beta = 0.5   # KG weight
config.fusion.gamma = 0.0  # LLM weight (disabled)

# Experiment name
config.experiment_name = "three_stream_v1"

print("Configuration:")
print(f"  Experiment: {config.experiment_name}")
print(f"  Device: {config.neural.device}")
print(f"  Batch size: {config.neural.batch_size}")
print(f"  Epochs: {config.neural.epochs}")
print(f"  LLM enabled: {config.llm.use_llm}")
print(f"  Fusion: α={config.fusion.alpha}, β={config.fusion.beta}")

---
## 3. Load Data

In [ ]:
# Create label mappings
label_to_idx, idx_to_label, num_labels = create_label_mappings(
    config.kg.techniques_json
)

print(f"Number of techniques: {num_labels}")
print(f"\nFirst 10 techniques:")
for i in range(min(10, num_labels)):
    print(f"  {i}: {idx_to_label[i]}")

In [ ]:
# Load data
data_loader = PropagandaDataLoader(config.data)
train_df, val_df, test_df = data_loader.load_all_splits()

print(f"\nData loaded:")
print(f"  Train: {len(train_df)} samples")
print(f"  Val: {len(val_df)} samples")
print(f"  Test: {len(test_df) if test_df is not None else 0} samples")

In [ ]:
# Preview data
print("\nSample from training data:")
sample = train_df.iloc[0]
print(f"ID: {sample['id']}")
print(f"Text: {sample['text'][:200]}...")
print(f"Image: {sample['image']}")
print(f"Labels: {sample['labels']}")
if 'caption' in sample:
    print(f"Caption: {sample['caption'][:200]}...")

---
## Choose Your Path

**Option A**: Use pretrained neural model (skip to section 5)

**Option B**: Train neural stream from scratch (continue to section 4)

---
## 4. Training (Option B)

Train the neural stream. KG and LLM streams don't need training.

In [ ]:
# Create dataloaders
train_loader, val_loader, test_loader, train_ds, val_ds, test_ds = create_dataloaders(
    config.data,
    label_to_idx,
    batch_size=config.neural.batch_size,
    use_caption=config.neural.use_caption,
    num_workers=0  # Use 0 for Colab
)

print(f"DataLoaders created:")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")

In [ ]:
# Initialize detector (this creates the neural stream)
detector = ThreeStreamPropagandaDetector(config)

# Note: The detector is initialized with random weights
# We'll train it in the next cell

In [ ]:
# Learn co-occurrence patterns from training data
# This improves KG stream performance
training_labels = train_df['labels'].tolist()
detector.kg_stream.update_co_occurrence_from_data(
    training_labels,
    min_support=5,
    pmi_threshold=0.5
)

In [ ]:
# Create trainer
trainer = NeuralStreamTrainer(
    detector.neural_stream,
    config.neural,
    idx_to_label,
    config.neural.device
)

print("Trainer ready!")

In [ ]:
# Train!
trainer.train(
    train_loader,
    val_loader,
    num_epochs=config.neural.epochs,
    checkpoint_dir=config.data.checkpoint_dir,
    save_best=True
)

# Save training history
history_path = Path(config.data.checkpoint_dir) / "training_history.json"
trainer.save_history(str(history_path))

---
## 5. Inference / Evaluation

Use the complete three-stream pipeline for prediction.

In [ ]:
# If you skipped training (Option A), load pretrained model:
# detector = ThreeStreamPropagandaDetector(config)
# detector.load_neural_checkpoint("path/to/checkpoint.pth")

# If you trained (Option B), detector is already ready!

In [ ]:
# Test on a single sample
from PIL import Image

# Get a sample from validation set
sample = val_df.iloc[0]
text = sample['text']
if 'caption' in sample and pd.notna(sample['caption']):
    text = f"{text} [SEP] {sample['caption']}"

img_path = Path(config.data.val_img_dir) / sample['image']
image = Image.open(img_path).convert('RGB')

# Predict
result = detector.predict(text, image, sample_id=sample['id'])

# Print results
detector.print_prediction(result, top_k=10)

In [ ]:
# Compare ground truth vs predictions
print("Ground Truth Labels:")
print(sample['labels'])

print("\nDetected Labels:")
print(result.detected_techniques)

print("\nCorrectly Detected:")
correct = set(sample['labels']) & set(result.detected_techniques)
print(correct)

print("\nMissed:")
missed = set(sample['labels']) - set(result.detected_techniques)
print(missed)

print("\nFalse Positives:")
fp = set(result.detected_techniques) - set(sample['labels'])
print(fp)

---
## 6. Batch Evaluation

In [ ]:
# Evaluate on full validation set (or subset for speed)
import numpy as np
from sklearn.metrics import classification_report, f1_score

# Select subset for faster testing (or use full dataset)
eval_df = val_df.head(100)  # Change to val_df for full evaluation

print(f"Evaluating on {len(eval_df)} samples...")

# Prepare data
texts = []
images = []
ground_truths = []

for idx, row in eval_df.iterrows():
    text = row['text']
    if 'caption' in row and pd.notna(row['caption']):
        text = f"{text} [SEP] {row['caption']}"
    
    img_path = Path(config.data.val_img_dir) / row['image']
    try:
        image = Image.open(img_path).convert('RGB')
    except:
        image = Image.new('RGB', (224, 224), 'white')
    
    texts.append(text)
    images.append(image)
    ground_truths.append(row['labels'])

# Batch predict
results = detector.predict_batch(texts, images, batch_size=16)

print("✓ Predictions complete!")

In [ ]:
# Compute metrics
y_true = []
y_pred = []

for gt, result in zip(ground_truths, results):
    # Convert to binary vectors
    true_vec = [1 if tech in gt else 0 for tech in detector.technique_names]
    pred_vec = [1 if tech in result.detected_techniques else 0 for tech in detector.technique_names]
    
    y_true.append(true_vec)
    y_pred.append(pred_vec)

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Overall metrics
f1_micro = f1_score(y_true, y_pred, average='micro', zero_division=0)
f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)

print("\n" + "="*60)
print("EVALUATION RESULTS")
print("="*60)
print(f"Micro F1: {f1_micro:.4f}")
print(f"Macro F1: {f1_macro:.4f}")
print("\nPer-Technique Report:")
print(classification_report(y_true, y_pred, target_names=detector.technique_names, zero_division=0))

In [ ]:
# Save predictions
results_path = Path(config.data.results_dir) / f"{config.experiment_name}_predictions.json"
detector.save_predictions(results, str(results_path), save_full_details=True)

print(f"Predictions saved to {results_path}")

---
## 7. Analysis & Visualization

In [ ]:
# Analyze stream contributions
import matplotlib.pyplot as plt

# Compare neural vs KG predictions
neural_scores = []
kg_scores = []
final_scores = []

for result in results[:50]:  # First 50 samples
    for tech in detector.technique_names:
        neural_scores.append(result.neural_predictions[tech])
        kg_scores.append(result.kg_predictions[tech])
        final_scores.append(result.final_predictions[tech])

# Plot
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(neural_scores, bins=50, alpha=0.7)
axes[0].set_title('Neural Predictions')
axes[0].set_xlabel('Score')

axes[1].hist(kg_scores, bins=50, alpha=0.7, color='green')
axes[1].set_title('KG Predictions')
axes[1].set_xlabel('Score')

axes[2].hist(final_scores, bins=50, alpha=0.7, color='orange')
axes[2].set_title('Final Fused Predictions')
axes[2].set_xlabel('Score')

plt.tight_layout()
plt.show()

In [ ]:
# Analyze disagreements between streams
disagreements = []

for result in results:
    for tech in detector.technique_names:
        neural = result.neural_predictions[tech]
        kg = result.kg_predictions[tech]
        disagreement = abs(neural - kg)
        
        if disagreement > 0.3:  # High disagreement
            disagreements.append({
                'sample_id': result.sample_id,
                'technique': tech,
                'neural': neural,
                'kg': kg,
                'disagreement': disagreement
            })

# Show top disagreements
disagreements_df = pd.DataFrame(disagreements)
if len(disagreements_df) > 0:
    top_disagreements = disagreements_df.nlargest(10, 'disagreement')
    print("Top 10 Stream Disagreements:")
    print(top_disagreements)
else:
    print("No significant disagreements found")

---
## 8. Export Results to Drive

In [ ]:
# Save final model checkpoint
final_checkpoint = Path(config.data.checkpoint_dir) / f"{config.experiment_name}_final.pth"
detector.save_neural_checkpoint(str(final_checkpoint))

# Save config
config_path = Path(config.data.results_dir) / f"{config.experiment_name}_config.json"
config.save(str(config_path))

print("\n✓ All results saved to Drive")
print(f"  Checkpoint: {final_checkpoint}")
print(f"  Config: {config_path}")
print(f"  Predictions: {results_path}")

---
## Summary

You have successfully:

✅ Loaded data from Google Drive

✅ Trained neural stream (or loaded pretrained)

✅ Applied knowledge graph reasoning

✅ Fused predictions from multiple streams

✅ Evaluated on validation set

✅ Saved all results back to Drive

### Next Steps:

- Optimize fusion weights on validation data
- Enable LLM stream (requires API key)
- Fine-tune thresholds per technique
- Analyze error cases
- Test on final test set